# 03 Fault prediction, evaluation, and geothermal candidates

This notebook trains fault models on the feature stacks from notebooks 01 and 02. It evaluates them with spatially blocked cross-validation. Then it extracts unmapped fault candidates and ranks them by structural settings that favor geothermal systems.

**Method summary.**

1. Engineer multi-scale neighborhood features.
2. Assign 20 km blocks to folds. Exclude a buffer around each test fold from training, so that fault traces that cross block edges do not leak.
3. Train LightGBM as a positive-unlabeled problem. Mapped faults are positives. Cells far from mapped faults are unlabeled and get a low weight. Cells in the band near mapped faults are left out of training, because the mapped trace position is uncertain there.
4. Score with a distance-weighted Tversky index (DTI). This index forgives predictions within 300 m of a mapped trace and penalizes missed faults more than false alarms (alpha = 0.2, beta = 0.8).
5. Optional: train a U-Net on a GPU and compare it with LightGBM on one held-out fold.
6. Extract candidate faults that are far from mapped faults. Rank them by prediction confidence, structural complexity (junctions and terminations), extensional strain, and shallow seismicity.

**Important limit.** The labels are incomplete. A high-confidence prediction far from a mapped fault counts as a false positive in the scores, but it can be a real, unmapped fault. The scores measure agreement with the current map. The candidate list in section 7 is the scientific output. Experts must review it.

**Annotations.** `NOTE:` marks a design decision or a heuristic that you can change.

**Environment.** `lightgbm`, `scikit-image`, `psutil`. Section 6 also needs `torch` and `segmentation-models-pytorch` and a GPU profile.

In [ ]:
import json
import os
import pathlib
import warnings

import fsspec
import geopandas as gpd
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import rasterio.features
import rioxarray  # noqa: F401
import xarray as xr
from rasterio.transform import from_origin
from scipy import ndimage
from shapely.geometry import shape
from skimage.morphology import skeletonize

## 0. Configuration

In [ ]:
STORE_ROOT = os.environ.get("GEMS_STORE", str(pathlib.Path.home() / "gems"))
CACHE_DIR = pathlib.Path(os.environ.get("GEMS_CACHE", "/tmp/gems_cache"))
REQUIRE_DEM = True             # NOTE: False lets the notebook run without notebook 02 output
DEM_NAME = "dem1m_features.zarr"
EXCLUDE_FEATURES = []          # base layer names to leave out

SEED = 42
N_FOLDS = 5
BLOCK_PX = 200                 # 20 km blocks at 100 m
BUFFER_PX = 10                 # 1 km exclusion buffer around each test fold

# Positive-unlabeled sampling. NOTE: heuristic values; test their effect.
NEG_MIN_DIST_M = 1000.0        # unlabeled cells must be at least this far from a mapped fault
UNLABELED_PER_POSITIVE = 5.0
UNLABELED_WEIGHT = 0.5

# Multi-scale features: smoothing scales in cells.
FEATURE_SCALES_PX = (3, 10)
MEMORY_FRACTION_LIMIT = 0.6    # stop if the feature stack needs more than this fraction of free RAM

LGB_PARAMS = {"objective": "binary", "learning_rate": 0.05, "num_leaves": 63,
              "min_data_in_leaf": 200, "feature_fraction": 0.8, "bagging_fraction": 0.8,
              "bagging_freq": 1, "lambda_l2": 1.0, "verbose": -1, "seed": SEED}
NUM_BOOST_ROUND, EARLY_STOP = 3000, 100

# Metric (same form as the GEMS challenge).
DTI_R_M, DTI_ALPHA, DTI_BETA = 300.0, 0.2, 0.8

# Post-processing grid searched on out-of-fold predictions.
PP_DILATE_PX = (0, 1)
PP_GAMMA = (0.5, 1.0, 2.0, 3.0)
PP_FLOOR = (0.0, 0.1, 0.2, 0.3)

# Candidates.
CAND_THRESHOLD = 0.5
CAND_MIN_DIST_M = 500.0        # farther than this from any mapped fault
CAND_MIN_LENGTH_M = 2000.0
COMPLEXITY_RADIUS_M = 2000.0
FAVOR_WEIGHTS = {"p_mean": 0.4, "complexity": 0.3, "gnss_dilatation": 0.2, "eq_logdens_shallow": 0.1}

RUN_UNET = False               # NOTE: needs a GPU profile

In [ ]:
CACHE_DIR.mkdir(parents=True, exist_ok=True)
fs, _ = fsspec.core.url_to_fs(STORE_ROOT)
def store_path(rel):
    return f"{STORE_ROOT.rstrip('/')}/{rel}"
fs.makedirs(store_path("predictions"), exist_ok=True)

feat = xr.open_zarr(store_path("features.zarr"))
lab = xr.open_zarr(store_path("labels.zarr"))
GRID = json.loads(feat.attrs["grid"])
H, W, RES = GRID["height"], GRID["width"], GRID["res"]
TRANSFORM = from_origin(GRID["xmin"], GRID["ymax"], RES, RES)

parts = [feat]
if fs.exists(store_path(DEM_NAME)):
    parts.append(xr.open_zarr(store_path(DEM_NAME)))
    print("Loaded", DEM_NAME)
elif REQUIRE_DEM:
    raise RuntimeError(f"{DEM_NAME} not found. Run notebook 02, or set REQUIRE_DEM = False.")
else:
    warnings.warn(f"{DEM_NAME} not found. Running without lidar features.")
ds = xr.merge(parts, join="exact", compat="override")

valid = ds["valid"].values.astype(bool)
fault = lab["fault"].values.astype(bool)
fault_dist = lab["fault_dist_m"].values
NON_FEATURES = {"valid", "spatial_ref"}
base_names = [n for n in ds.data_vars if n not in NON_FEATURES and n not in EXCLUDE_FEATURES]
print(f"{len(base_names)} base layers; valid area {valid.mean():.1%}; fault cells in valid area {int((fault & valid).sum()):,}")

## 1. Distance-weighted Tversky index

This is a local implementation of the metric from the GEMS problem description. `M(g)` is the best kernel-weighted prediction within `R` of ground-truth cell `g`. `TP = sum M(g)`, `FN = sum (1 - M(g))`, and `FP = sum p(x) (1 - k(d(x)))`, where `d(x)` is the distance from `x` to the nearest ground-truth cell and `k(d) = max(1 - d / R, 0)`.

In [ ]:
def _shift(a, dy, dx):
    """out[y, x] = a[y + dy, x + dx], zero outside."""
    out = np.zeros_like(a)
    h, w = a.shape
    ys, yd = (slice(dy, h), slice(0, h - dy)) if dy >= 0 else (slice(0, h + dy), slice(-dy, h))
    xs, xd = (slice(dx, w), slice(0, w - dx)) if dx >= 0 else (slice(0, w + dx), slice(-dx, w))
    out[yd, xd] = a[ys, xs]
    return out

def dti(p, g, mask=None, res=100.0, R=300.0, alpha=0.2, beta=0.8, eps=1e-9):
    p = np.nan_to_num(np.asarray(p, np.float64))
    g = np.asarray(g, bool)
    if mask is not None:
        p = np.where(mask, p, 0.0)
        g = g & mask
    r = int(np.floor(R / res))
    M = np.zeros_like(p)
    for dy in range(-r, r + 1):
        for dx in range(-r, r + 1):
            k = 1.0 - res * np.hypot(dy, dx) / R
            if k > 0:
                np.maximum(M, _shift(p, dy, dx) * k, out=M)
    tp = float(M[g].sum())
    fn = float((1.0 - M[g]).sum())
    if g.any():
        kd = np.clip(1.0 - ndimage.distance_transform_edt(~g) * res / R, 0.0, None)
    else:
        kd = np.zeros_like(p)
    fp = float((p * (1.0 - kd))[p > 0].sum())
    return tp / (tp + alpha * fp + beta * fn + eps), {"TP": tp, "FP": fp, "FN": fn}

## 2. Features

For every base layer the notebook adds NaN-aware Gaussian smooths at each scale in `FEATURE_SCALES_PX`, the residual (layer minus smooth), and the gradient magnitude of the finest smooth. The orientation layers (`*_sin2`, `*_cos2`) and coverage layers only get the finest smooth.

In [ ]:
def nan_gaussian(a, sigma):
    m = np.isfinite(a)
    num = ndimage.gaussian_filter(np.where(m, a, 0).astype(np.float32), sigma)
    den = ndimage.gaussian_filter(m.astype(np.float32), sigma)
    with np.errstate(invalid="ignore", divide="ignore"):
        out = num / den
    out[den < 1e-3] = np.nan
    return out.astype(np.float32)

def engineer(name, a, scales):
    out = {name: a}
    light = name.endswith(("_sin2", "_cos2")) or name.endswith("cov")
    for i, s in enumerate(scales):
        sm = nan_gaussian(a, s)
        out[f"{name}__g{s}"] = sm
        if light:
            break
        out[f"{name}__r{s}"] = (a - sm).astype(np.float32)
        if i == 0:
            gr, gc = np.gradient(sm)
            out[f"{name}__grad{s}"] = np.hypot(gr, gc).astype(np.float32)
    return out

In [ ]:
n_est = sum(1 + (1 if n.endswith(("_sin2", "_cos2", "cov")) else 2 * len(FEATURE_SCALES_PX) + 1) for n in base_names)
need = n_est * H * W * 4
avail = psutil.virtual_memory().available
print(f"About {n_est} features, {need / 1e9:.1f} GB; {avail / 1e9:.1f} GB available")
if need > MEMORY_FRACTION_LIMIT * avail:
    raise MemoryError("Feature stack is too large. Use fewer scales, EXCLUDE_FEATURES, or a larger server.")

F = {}
for n in base_names:
    F.update(engineer(n, ds[n].values.astype(np.float32), FEATURE_SCALES_PX))
feature_names = list(F)
print(len(feature_names), "features")

def gather(rows, cols):
    return np.column_stack([F[n][rows, cols] for n in feature_names])

## 3. Spatial folds and positive-unlabeled sampling

Blocks are dealt to folds in order of their fault-cell count, so each fold gets a similar share of faults.

In [ ]:
rng = np.random.default_rng(SEED)
nby, nbx = int(np.ceil(H / BLOCK_PX)), int(np.ceil(W / BLOCK_PX))
block_id = (np.arange(H)[:, None] // BLOCK_PX) * nbx + (np.arange(W)[None, :] // BLOCK_PX)
counts = np.bincount(block_id[fault & valid], minlength=nby * nbx).astype(float)
order = np.argsort(counts + rng.random(counts.size) * 0.5)[::-1]
fold_of_block = np.empty(nby * nbx, int)
fold_of_block[order] = np.arange(order.size) % N_FOLDS
fold = fold_of_block[block_id]

def sample_training(train_mask, rng):
    pr, pc = np.nonzero(train_mask & fault)
    ur, uc = np.nonzero(train_mask & (fault_dist > NEG_MIN_DIST_M))
    n_u = min(len(ur), int(len(pr) * UNLABELED_PER_POSITIVE))
    pick = rng.choice(len(ur), n_u, replace=False)
    rows = np.concatenate([pr, ur[pick]])
    cols = np.concatenate([pc, uc[pick]])
    y = np.concatenate([np.ones(len(pr)), np.zeros(n_u)]).astype(np.float32)
    w = np.concatenate([np.ones(len(pr)), np.full(n_u, UNLABELED_WEIGHT)]).astype(np.float32)
    return rows, cols, y, w

def fit_lgb(rows, cols, y, w, rng, num_round=None):
    X = gather(rows, cols)
    if num_round is None:
        # Early stopping on 15 % of the training blocks.
        blocks = np.unique(block_id[rows, cols])
        es_blocks = rng.choice(blocks, max(1, int(0.15 * blocks.size)), replace=False)
        es = np.isin(block_id[rows, cols], es_blocks)
        dtr = lgb.Dataset(X[~es], y[~es], weight=w[~es], feature_name=feature_names, free_raw_data=True)
        dva = lgb.Dataset(X[es], y[es], weight=w[es], reference=dtr)
        return lgb.train(LGB_PARAMS, dtr, NUM_BOOST_ROUND, valid_sets=[dva],
                         callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False), lgb.log_evaluation(0)])
    return lgb.train(LGB_PARAMS, lgb.Dataset(X, y, weight=w, feature_name=feature_names), num_round)

def predict_mask(model, mask, chunk=2_000_000):
    rows, cols = np.nonzero(mask)
    out = np.full((H, W), np.nan, np.float32)
    for i in range(0, rows.size, chunk):
        r, c = rows[i:i + chunk], cols[i:i + chunk]
        out[r, c] = model.predict(gather(r, c), num_iteration=model.best_iteration or None)
    return out

## 4. Cross-validated LightGBM

In [ ]:
oof = np.full((H, W), np.nan, np.float32)
fold_scores, best_iters = [], []
for k in range(N_FOLDS):
    test = (fold == k) & valid
    near_test = ndimage.binary_dilation(fold == k, iterations=BUFFER_PX)
    train = valid & ~near_test
    rows, cols, y, w = sample_training(train, rng)
    model = fit_lgb(rows, cols, y, w, rng)
    best_iters.append(model.best_iteration)
    oof[test] = predict_mask(model, test)[test]
    score, parts_ = dti(oof, fault, mask=test, res=RES, R=DTI_R_M, alpha=DTI_ALPHA, beta=DTI_BETA)
    fold_scores.append(score)
    print(f"fold {k}: {int(y.sum()):,} pos, {int((y == 0).sum()):,} unlabeled, "
          f"{model.best_iteration} trees, DTI {score:.3f}  {parts_}")
print(f"Mean DTI (raw probabilities): {np.mean(fold_scores):.3f} +/- {np.std(fold_scores):.3f}")

In [ ]:
def postprocess(p, dilate, gamma, floor):
    q = np.nan_to_num(p)
    if dilate:
        q = ndimage.maximum_filter(q, size=2 * dilate + 1)
    q = q ** gamma
    q[q < floor] = 0.0
    return np.where(valid, q, np.nan).astype(np.float32)

# NOTE: the settings are tuned on the same out-of-fold map that is scored, so the tuned
# score is slightly optimistic. Compare it with the raw score above.
grid_results = []
for d in PP_DILATE_PX:
    for g_ in PP_GAMMA:
        for fl in PP_FLOOR:
            s = np.mean([dti(postprocess(oof, d, g_, fl), fault, mask=(fold == k) & valid, res=RES,
                             R=DTI_R_M, alpha=DTI_ALPHA, beta=DTI_BETA)[0] for k in range(N_FOLDS)])
            grid_results.append({"dilate": d, "gamma": g_, "floor": fl, "dti": s})
pp = pd.DataFrame(grid_results).sort_values("dti", ascending=False)
display(pp.head(10))
BEST_PP = pp.iloc[0][["dilate", "gamma", "floor"]].to_dict()
BEST_PP["dilate"] = int(BEST_PP["dilate"])
print("Best post-processing:", BEST_PP)

## 5. Final model, feature contributions, and prediction map

The final model trains on all folds with the median number of trees from cross-validation. LightGBM computes SHAP contributions natively with `pred_contrib=True`.

In [ ]:
rows, cols, y, w = sample_training(valid, rng)
final = fit_lgb(rows, cols, y, w, rng, num_round=int(np.median(best_iters)))

sub = rng.choice(rows.size, min(50_000, rows.size), replace=False)
contrib = final.predict(gather(rows[sub], cols[sub]), pred_contrib=True)[:, :-1]
imp = pd.Series(np.abs(contrib).mean(axis=0), index=feature_names).sort_values(ascending=False)
imp.head(25)[::-1].plot.barh(figsize=(7, 8), title="Mean |SHAP| (top 25)")
plt.tight_layout()

p_raw = predict_mask(final, valid)
p_final = postprocess(p_raw, **BEST_PP)

def write_cog(arr, rel):
    da = xr.DataArray(arr, coords={"y": ds.y, "x": ds.x}, dims=("y", "x")).rio.write_crs(GRID["crs"])
    da = da.rio.write_transform(TRANSFORM).rio.write_nodata(np.nan)
    local = CACHE_DIR / pathlib.Path(rel).name
    da.rio.to_raster(local, driver="COG", dtype="float32")
    fs.put(str(local), store_path(rel))

write_cog(p_raw, "predictions/fault_prob_raw.tif")
write_cog(p_final, "predictions/fault_prob_post.tif")
write_cog(np.where(valid, oof, np.nan).astype(np.float32), "predictions/fault_prob_oof.tif")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(p_final, cmap="magma", vmin=0, vmax=1)
ax.contour(fault, levels=[0.5], colors="cyan", linewidths=0.3)
ax.set_title("Fault probability (post-processed); cyan = mapped Quaternary faults")
ax.set_axis_off()

## 6. Optional: U-Net on a GPU

This section trains a U-Net on the base layers of the training folds and compares it with LightGBM on fold `HOLDOUT_FOLD`. The loss is a masked Tversky loss with the metric weights plus binary cross-entropy. The loss ignores the uncertain band near mapped faults, as the LightGBM sampling does.

Set `RUN_UNET = True` in section 0 to run it. The cell stops if no GPU is available.

In [ ]:
HOLDOUT_FOLD, PATCH, BATCH, STEPS, EPOCHS = 0, 128, 16, 400, 15

def run_unet():
    import torch
    import torch.nn.functional as TF
    import segmentation_models_pytorch as smp
    if not torch.cuda.is_available():
        raise RuntimeError("RUN_UNET is True, but no GPU is available. Select a GPU profile.")
    dev = torch.device("cuda")

    train_area = valid & ~ndimage.binary_dilation(fold == HOLDOUT_FOLD, iterations=BUFFER_PX)
    chans = []
    for n in base_names:
        a = ds[n].values.astype(np.float32)
        v = a[train_area & np.isfinite(a)]
        med, iqr = (np.median(v), np.subtract(*np.percentile(v, [75, 25])) + 1e-6) if v.size else (0.0, 1.0)
        chans.append(np.clip(np.nan_to_num((a - med) / iqr), -10, 10))
    chans.append(valid.astype(np.float32))
    X = np.stack(chans)
    Y = fault.astype(np.float32)
    uncertain = (fault_dist > 0) & (fault_dist <= NEG_MIN_DIST_M)
    Mtrain = (train_area & ~uncertain).astype(np.float32)

    centers_pos = np.argwhere(train_area & fault)
    centers_any = np.argwhere(train_area)
    half = PATCH // 2
    def batch():
        xs, ys, ms = [], [], []
        for _ in range(BATCH):
            c = centers_pos[rng.integers(len(centers_pos))] if rng.random() < 0.5 else centers_any[rng.integers(len(centers_any))]
            r0 = int(np.clip(c[0] - half, 0, H - PATCH)); c0 = int(np.clip(c[1] - half, 0, W - PATCH))
            xs.append(X[:, r0:r0 + PATCH, c0:c0 + PATCH]); ys.append(Y[r0:r0 + PATCH, c0:c0 + PATCH])
            ms.append(Mtrain[r0:r0 + PATCH, c0:c0 + PATCH])
        t = lambda a: torch.from_numpy(np.stack(a)).to(dev)
        return t(xs), t(ys)[:, None], t(ms)[:, None]

    def loss_fn(logits, y, m, a=DTI_ALPHA, b=DTI_BETA):
        p = torch.sigmoid(logits)
        tp, fp, fn = (p * y * m).sum(), (p * (1 - y) * m).sum(), ((1 - p) * y * m).sum()
        tversky = 1 - (tp + 1) / (tp + a * fp + b * fn + 1)
        bce = (TF.binary_cross_entropy_with_logits(logits, y, reduction="none") * m).sum() / m.sum().clamp(min=1)
        return tversky + bce

    net = smp.Unet("resnet34", encoder_weights=None, in_channels=X.shape[0], classes=1).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * STEPS)
    for ep in range(EPOCHS):
        net.train(); tot = 0.0
        for _ in range(STEPS):
            xb, yb, mb = batch()
            loss = loss_fn(net(xb), yb, mb)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            tot += loss.item()
        print(f"epoch {ep}: loss {tot / STEPS:.4f}")

    # Sliding-window prediction over the whole grid.
    net.eval(); win, stride = 256, 192
    Hp, Wp = int(np.ceil((H - win) / stride)) * stride + win, int(np.ceil((W - win) / stride)) * stride + win
    Xp = np.pad(X, ((0, 0), (0, max(Hp - H, 0)), (0, max(Wp - W, 0))), mode="reflect")
    acc = np.zeros(Xp.shape[1:], np.float32); cnt = np.zeros_like(acc)
    with torch.no_grad():
        for r0 in range(0, Xp.shape[1] - win + 1, stride):
            for c0 in range(0, Xp.shape[2] - win + 1, stride):
                xb = torch.from_numpy(Xp[None, :, r0:r0 + win, c0:c0 + win]).to(dev)
                acc[r0:r0 + win, c0:c0 + win] += torch.sigmoid(net(xb))[0, 0].cpu().numpy()
                cnt[r0:r0 + win, c0:c0 + win] += 1
    return np.where(valid, (acc / np.maximum(cnt, 1))[:H, :W], np.nan).astype(np.float32)

if RUN_UNET:
    p_unet = run_unet()
    ho = (fold == HOLDOUT_FOLD) & valid
    for name, p in [("LightGBM (OOF)", oof), ("U-Net", p_unet), ("Mean of both", np.nanmean([oof, p_unet], axis=0))]:
        print(f"{name:16s} DTI on fold {HOLDOUT_FOLD}: {dti(p, fault, mask=ho, res=RES, R=DTI_R_M, alpha=DTI_ALPHA, beta=DTI_BETA)[0]:.3f}")
    write_cog(p_unet, "predictions/fault_prob_unet.tif")

## 7. Unmapped fault candidates and geothermal ranking

A candidate is a connected region where the post-processed probability is at least `CAND_THRESHOLD` and the distance to the nearest mapped fault is more than `CAND_MIN_DIST_M`. The notebook skeletonizes each candidate to measure its length and strike.

**Structural complexity.** Geothermal systems in the Great Basin favor step-overs, fault terminations, fault intersections, and accommodation zones. The notebook skeletonizes the combined network of mapped faults and candidates. It counts junctions (three or more neighbors) and endpoints (one neighbor) within `COMPLEXITY_RADIUS_M` of each candidate.

**Score.** The score is a weighted sum of percentile ranks (`FAVOR_WEIGHTS`). `NOTE:` this is a screening heuristic, not a calibrated favorability model. Compare the list with the INGENIOUS favorability maps.

In [ ]:
EIGHT = np.ones((3, 3), int)

cand_bin = (np.nan_to_num(p_final) >= CAND_THRESHOLD) & (fault_dist > CAND_MIN_DIST_M) & valid
cand_lab, n_cand = ndimage.label(cand_bin, structure=EIGHT)
cand_skel = skeletonize(cand_bin)

network = skeletonize(fault | cand_bin)
nbrs = ndimage.convolve(network.astype(int), EIGHT, mode="constant") - network
junction = network & (nbrs >= 3)
endpoint = network & (nbrs == 1)
win = 2 * int(COMPLEXITY_RADIUS_M / RES) + 1
complexity_map = (ndimage.uniform_filter(junction.astype(np.float32), win) +
                  ndimage.uniform_filter(endpoint.astype(np.float32), win)) * win * win

xs = GRID["xmin"] + RES * (np.arange(W) + 0.5)
ys = GRID["ymax"] - RES * (np.arange(H) + 0.5)
recs = []
for i, sl in enumerate(ndimage.find_objects(cand_lab), start=1):
    comp = cand_lab[sl] == i
    sk = cand_skel[sl] & comp
    length = sk.sum() * RES                      # NOTE: approximate; diagonal steps count as RES
    if length < CAND_MIN_LENGTH_M:
        continue
    rr, cc = np.nonzero(sk)
    px, py = xs[cc + sl[1].start], ys[rr + sl[0].start]
    if px.size >= 2:
        evals, evecs = np.linalg.eigh(np.cov(np.vstack([px, py])))
        vx, vy = evecs[:, -1]
        strike = float(np.degrees(np.arctan2(vx, vy)) % 180.0)
    else:
        strike = np.nan
    rec = {"cand_id": i, "length_m": float(length), "strike_deg": strike,
           "p_mean": float(np.nanmean(p_final[sl][comp])), "p_max": float(np.nanmax(p_final[sl][comp])),
           "complexity": float(complexity_map[sl][comp].max()),
           "dist_to_mapped_m": float(fault_dist[sl][comp].min())}
    for extra in ("gnss_dilatation", "eq_logdens_shallow"):
        if extra in ds:
            rec[extra] = float(np.nanmean(ds[extra].values[sl][comp]))
    recs.append(rec)

cands = pd.DataFrame(recs)
print(f"{n_cand} candidate regions, {len(cands)} longer than {CAND_MIN_LENGTH_M / 1000:.0f} km")

In [ ]:
score = np.zeros(len(cands))
used = 0.0
for col, wgt in FAVOR_WEIGHTS.items():
    if col not in cands or cands[col].isna().all():
        warnings.warn(f"{col} is not available. It is left out of the score.")
        continue
    score += wgt * cands[col].rank(pct=True).fillna(0.0).values
    used += wgt
cands["favorability"] = score / max(used, 1e-9)
cands = cands.sort_values("favorability", ascending=False).reset_index(drop=True)

keep = np.isin(cand_lab, cands.cand_id.values)
polys = [(shape(geom), int(v)) for geom, v in rasterio.features.shapes(
    cand_lab.astype(np.int32), mask=keep, transform=TRANSFORM, connectivity=8)]
gpoly = gpd.GeoDataFrame({"cand_id": [v for _, v in polys]}, geometry=[g for g, _ in polys], crs=GRID["crs"])
gpoly = gpoly.dissolve("cand_id").reset_index().merge(cands, on="cand_id")
gpoly = gpoly.sort_values("favorability", ascending=False)

local = CACHE_DIR / "fault_candidates.gpkg"
gpoly.to_file(local, driver="GPKG")
fs.put(str(local), store_path("predictions/fault_candidates.gpkg"))
display(cands.head(20))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(np.where(valid, 1.0, np.nan), cmap="Greys", vmin=0, vmax=3)
ax.contour(fault, levels=[0.5], colors="0.3", linewidths=0.3)
top = np.isin(cand_lab, cands.cand_id.head(25).values)
ax.imshow(np.where(top, 1.0, np.nan), cmap="autumn", vmin=0, vmax=1)
ax.set_title("Top 25 unmapped fault candidates (red) with mapped Quaternary faults (grey)")
ax.set_axis_off()

## 8. Next steps

* Review the top candidates against the 1 m hillshade, the GeoDAWN magnetic derivatives, and published geologic maps.
* Compare `fault_candidates.gpkg` with the INGENIOUS favorability maps and known geothermal systems.
* Test the effect of the positive-unlabeled settings (`NEG_MIN_DIST_M`, `UNLABELED_WEIGHT`) and the block size on the fold scores.
* Remove one feature group at a time (lidar, magnetics, GNSS, seismicity) and compare the fold scores. This shows which data sets carry the signal.